# AMRVAC DataConstrain Pipeline

Single-frame workflow: magnetic input -> potential field -> magnetofrictional relaxation -> fixed-boundary DataConstrained MHD. This notebook prepares data and AMRVAC cases; run AMRVAC commands in a terminal.


## 1. Setup

Locate AMRVAC and check the Python environment. **Required for all inputs:** `numpy`, `matplotlib`, and `astropy`. **Additionally required for raw HMI:** `scipy` and `sunpy`. Install them in the environment used by this Jupyter kernel, for example with `%pip install numpy scipy matplotlib astropy sunpy`, then restart the kernel. VTK, tqdm, and DAVE4VM are not required by this single-frame workflow.

**Required:** export `AMRVAC_DIR`. **Optional:** set the fallback below only when that environment variable is unavailable.


In [ ]:
import os
import sys
from pathlib import Path

USER_AMRVAC_DIR = None  # OPTIONAL: set Path('/path/to/amrvac') only if AMRVAC_DIR is not exported.

amrvac_dir = os.environ.get('AMRVAC_DIR') or USER_AMRVAC_DIR
if amrvac_dir is None:
    raise EnvironmentError('Export AMRVAC_DIR before starting Jupyter, or set USER_AMRVAC_DIR above.')
AMRVAC_ROOT = Path(amrvac_dir).expanduser().resolve()
PYTOOLS = AMRVAC_ROOT / 'tools/python'
if not (PYTOOLS / 'amrvac_pytools').is_dir():
    raise ModuleNotFoundError(f'Cannot locate amrvac_pytools under {PYTOOLS}. Check AMRVAC_DIR.')
if str(PYTOOLS) not in sys.path:
    sys.path.insert(0, str(PYTOOLS))

from amrvac_pytools.datadriven import (
    check_data_driven_dependencies, create_data_constrain_workflow, stage_grid_config
)

check_data_driven_dependencies()
print(f'AMRVAC root: {AMRVAC_ROOT}')


## 2. Input And Project

Set the magnetic-data directory and inspect the first frame. Raw HMI is shown as $B_{\rm LOS}$ before CEA remapping; CEA input is shown as $B_r$. **Required:** input directory. **Recommended:** project directory and resolution levels. **Optional:** preview limit and block sizes.


In [ ]:
INPUT_DIR = Path('/path/to/raw_HMI')  # REQUIRED
USER_PROJECT_DIR = None  # RECOMMENDED: set a portable project directory; None uses AMRVAC_ROOT / 'DrivenFieldProject'.
PREVIEW_BMAX = 1000.0  # OPTIONAL: symmetric color limit [G]; None selects it automatically.

RELAXATION_BOUNDARY_REDUCTION_LEVEL = 3  # RECOMMENDED: Potential/MFR boundary reduction; 2 means half resolution.
EVOLUTION_BOUNDARY_REDUCTION_LEVEL = 2  # RECOMMENDED: DataConstrained boundary reduction; 1 keeps the selected resolution.
EVOLUTION_AMRVAC_REFINEMENT_LEVEL = 1  # RECOMMENDED: finest DataConstrained AMR level.
AMRVAC_BLOCK_SIZES = (12, 14, 16, 18, 20)  # OPTIONAL: allowed AMRVAC block sizes.

workflow = create_data_constrain_workflow(
    amrvac_root=AMRVAC_ROOT,
    project_dir=USER_PROJECT_DIR,
    input_dir=INPUT_DIR,
    relaxation_grid=stage_grid_config(
        RELAXATION_BOUNDARY_REDUCTION_LEVEL, 1, AMRVAC_BLOCK_SIZES
    ),
    evolution_grid=stage_grid_config(
        EVOLUTION_BOUNDARY_REDUCTION_LEVEL, EVOLUTION_AMRVAC_REFINEMENT_LEVEL, AMRVAC_BLOCK_SIZES
    ),
)
workflow.preview_input(bmax=PREVIEW_BMAX)
print(workflow.input_report())


## 3. Region And Grid Plan

Select one master region. With `REGION_MODE='auto'`, raw HMI uses `RAW_HMI_CEA_PATCH`, while an existing SHARP/CEA map uses `SHARP_PIXEL_WINDOW`; the inactive parameter is ignored. Set `fixed_cea` to force the CEA patch for either input type, or `pixel_window` to force a pixel crop for SHARP/CEA only. Automatic block-compatible trimming is optional.


In [ ]:
REGION_MODE = 'auto'  # RECOMMENDED: auto, fixed_cea, or pixel_window.
RAW_HMI_CEA_PATCH = {  # Active default: auto raw HMI, or fixed_cea with either input type.
    'center_lon': 7.0,
    'center_lat': 13.0,
    'width_degree': 40.0,
    'height_degree': 30.0,
    'resolution_degree': 0.03,
}
SHARP_PIXEL_WINDOW = None  # Used for auto SHARP/CEA or pixel_window; None selects the full map.
# Example crop: {'x0': 100, 'y0': 0, 'nx': 512, 'ny': 512}. Inactive settings are ignored.
AUTO_TRIM_TO_AMRVAC_BLOCKS = True  # OPTIONAL: minimally trim incompatible dimensions when True.

workflow.plan_region(
    region_mode=REGION_MODE,
    raw_hmi_cea_patch=RAW_HMI_CEA_PATCH,
    sharp_pixel_window=SHARP_PIXEL_WINDOW,
    auto_trim=AUTO_TRIM_TO_AMRVAC_BLOCKS,
)
workflow.plot_region(bmax=PREVIEW_BMAX)
print(workflow.region_report())


## 4. Prepare The Reference Frame And Initial-Field Cases

Prepare only the selected frame and stage PotentialField plus MagnetofrictionalRelaxation. **Recommended:** confirm snapshot, potential method, and MFR iteration limits. **Optional:** remap, ghost-cell, preprocessing, and solver tuning.


In [ ]:
SNAPSHOT_INDEX = 0  # RECOMMENDED: confirm this when INPUT_DIR contains a sequence; indexing starts at 0.

CEA_REMAP_OPTIONS = {  # OPTIONAL: defaults reproduce the SHARP-like remap.
    'sampling_mode': 'sharp',
    'oversample_resolution_degree': 0.01,
    'smooth_sigma_degree': 0.01,
}
BOUNDARY_OPTIONS = {
    'nghost': 2,  # OPTIONAL: ghost cells written around the physical magnetogram.
    'preprocess': False,  # OPTIONAL: Cartesian boundary preprocessing.
    'vmax': 500.0,  # OPTIONAL: boundary quicklook color limit [G].
}
POTENTIAL_OPTIONS = {
    'potential_field_method': 'fft',  # RECOMMENDED: fft or green.
    'fft_padding_factor': 2,  # OPTIONAL: FFT horizontal padding.
    'lalpha': 0.0,  # OPTIONAL: 0 gives a potential field; nonzero gives constant-alpha LFFF.
    'fft_top_boundary': 'open',  # OPTIONAL: open or closed.
    # 'potential_zshift_Mm': 3.0,  # OPTIONAL: used only by the Green-function method.
}
MFR_OPTIONS = {
    'mf_it_max': 100000,  # RECOMMENDED: maximum relaxation iterations.
    'mf_ditsave': 5000,  # RECOMMENDED: restart snapshot interval.
    'mf_cc': 0.5,  # OPTIONAL
    'mf_cy': 0.2,  # OPTIONAL
    'mf_cdivb': 0.01,  # OPTIONAL
}

workflow.prepare_initial_field(
    snapshot_index=SNAPSHOT_INDEX,
    cea_remap_options=CEA_REMAP_OPTIONS,
    potential_options=POTENTIAL_OPTIONS,
    mfr_options=MFR_OPTIONS,
    **BOUNDARY_OPTIONS,
)
print(workflow.initial_field_report())


## 5. Run PotentialField And MagnetofrictionalRelaxation

Run these commands in a terminal, in order. **Recommended:** adjust the MPI process count for the target machine.


In [ ]:
NPROC = 4  # RECOMMENDED: choose a suitable MPI process count for your machine.

print(workflow.initial_field_commands(nproc=NPROC))


## 6. Select The Relaxed Field And Stage DataConstrained

Run this after magnetofriction finishes. **Recommended:** choose the MHD model. **Optional:** override the automatic restart and model-dependent atmosphere defaults. A relaxed-table atmosphere additionally requires its file path.


In [ ]:
SELECTED_RESTART_NUMBER = None  # OPTIONAL: e.g. 4 selects data_driven_mfr0004.dat; None uses diagnostics.
PLOT_LORENTZ_FORCE = False  # OPTIONAL: True overlays the Lorentz force on a second y-axis.
DATA_CONSTRAINED_OPTIONS = {
    'mhd_model': 'zero_beta',  # RECOMMENDED: zero_beta, isothermal, adiabatic, or thermodynamic.
    'atmosphere_model': None,  # OPTIONAL: None chooses uniform/corona/corona/chromosphere by MHD model.
    'atmosphere_source': 'hydrostatic',  # OPTIONAL: hydrostatic or relaxed_table.
    # 'relaxed_atmosphere_file': Path('/path/to/atmosphere.dat'),  # REQUIRED only for relaxed_table.
    # 'coronal_temperature_k': 1.0e6,  # OPTIONAL
    # 'rho_reference_numberdensity_cm3': 1.0e9,  # OPTIONAL
    # 'temperature_curve': 'AL-C7',  # OPTIONAL for chromosphere.
    # 'heating_amplitude_cgs': 1.0e-4,  # OPTIONAL for thermodynamic MHD.
    # 'heating_scale_height_cm': 5.0e9,  # OPTIONAL for thermodynamic MHD.
}

workflow.analyze_and_stage_data_constrained(
    selected_restart_number=SELECTED_RESTART_NUMBER,
    plot_lorentz_force=PLOT_LORENTZ_FORCE,
    data_constrained_options=DATA_CONSTRAINED_OPTIONS,
)
print(workflow.data_constrained_report())


## 7. Run DataConstrained

Run the staged fixed-boundary MHD case in a terminal.


In [ ]:
print(workflow.data_constrained_commands(nproc=NPROC))
